Building it on top of the existing dim_employee_scd2 table

In [0]:
%sql
CREATE OR REPLACE TABLE ibm_hr.silver.dim_employee_scd6 AS
SELECT
  employee_sk,
  employee_id,
  Age, Gender, MaritalStatus,
  Department,
  CAST(NULL AS STRING) AS previous_department,
  Department AS current_department,
  JobRole, JobLevel, Education, EducationField, BusinessTravel, OverTime, Attrition,
  from_date, to_date, in_use_flag
FROM ibm_hr.silver.dim_employee_scd2

In [0]:
spark.sql("SELECT employee_sk, employee_id, Department, previous_department, current_department, from_date, to_date, in_use_flag FROM ibm_hr.silver.dim_employee_scd6 WHERE employee_id = 1 ORDER BY from_date").show()

Simulating a real Type 6 change and see all three mechanisms fire at once. Employee 1 moves from Sales to Human Resources.

In [0]:
%sql
MERGE INTO ibm_hr.silver.dim_employee_scd6 AS target
USING (SELECT 1 AS employee_id, 'Human Resources' AS new_department) AS source
ON target.employee_id = source.employee_id AND target.in_use_flag = true
WHEN MATCHED AND target.Department <> source.new_department THEN
  UPDATE SET
    target.to_date = CAST('2026-09-01' AS DATE),
    target.in_use_flag = false

Inserting the new active row, with previous_department set (Type 3):

In [0]:
%sql
INSERT INTO ibm_hr.silver.dim_employee_scd6
SELECT
  (SELECT MAX(employee_sk) + 1 FROM ibm_hr.silver.dim_employee_scd6),
  employee_id, Age, Gender, MaritalStatus,
  'Human Resources' AS Department,
  Department AS previous_department,
  'Human Resources' AS current_department,
  JobRole, JobLevel, Education, EducationField, BusinessTravel, OverTime, Attrition,
  CAST('2026-09-01' AS DATE) AS from_date,
  CAST(NULL AS DATE) AS to_date,
  TRUE AS in_use_flag
FROM ibm_hr.silver.dim_employee_scd6
WHERE employee_id = 1 AND in_use_flag = false
ORDER BY to_date DESC
LIMIT 1

Updating current_department on ALL rows for this employee, including the old closed ones

In [0]:
%sql
UPDATE ibm_hr.silver.dim_employee_scd6
SET current_department = 'Human Resources'
WHERE employee_id = 1

In [0]:
spark.sql("SELECT employee_sk, Department, previous_department, current_department, from_date, to_date, in_use_flag FROM ibm_hr.silver.dim_employee_scd6 WHERE employee_id = 1 ORDER BY from_date").show()

In [0]:
spark.sql("""
SELECT employee_sk, Department, to_date, in_use_flag
FROM ibm_hr.silver.dim_employee_scd6
WHERE employee_id = 1
ORDER BY to_date DESC
""").show()

In [0]:
%sql
UPDATE ibm_hr.silver.dim_employee_scd6
SET previous_department = 'Sales'
WHERE employee_sk = 1471

In [0]:
spark.sql("SELECT employee_sk, Department, previous_department, current_department, from_date, to_date, in_use_flag FROM ibm_hr.silver.dim_employee_scd6 WHERE employee_id = 1 ORDER BY from_date").show()